# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env(".env")

OPENAI_API_KEY=****ywsA
DEEPSEEK_API_KEY=****c1e4
DEEPSEEK_BASE_URL=****.com
LANGSMITH_API_KEY=****b63c
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


## Human👨‍💻 and AI 🤖 Messages

In [2]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="deepseek-chat",
    system_prompt="你是一名全栈喜剧演员"
)

In [ ]:
# 这一段看第二段导入的包,具体作用为将该信息标记为用户消息
# 具体的转换格式为 "你好,最近怎么样"
# ->{'messages':[{'role':user, 'content':"你好,最近怎么样"}]}
# 不过这里具体调试后格式为[HumanMessage(content="你好,最近怎么样")]
human_msg = HumanMessage("你好,最近怎么样")
# 将消息作为输入发给agent
result = agent.invoke({"messages": [human_msg]})

In [ ]:
# 将消息的最后一个元素(一般都是ai回答)的content内容打印出来
print(result["messages"][-1].content)

In [ ]:
# 打印最后一个元素的类型
print(type(result["messages"][-1]))
# <class 'langchain_core.messages.ai.AIMessage'>

In [ ]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [7]:
agent = create_agent(
    model="deepseek-chat",
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [ ]:
#这里使用通用的格式化消息,和其他效果是一样的,不过更加通用
result = agent.invoke(
    {"messages": {"role": "user", "content": "写一首关于短跑运动员的俳句"}}
)
print(result["messages"][-1].content)
#消息也有多种形式,sys指系统(系统提示词),user是用户,ass是ai
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]

#### Dictionaries

In [ ]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "写一首关于短跑运动员的俳句"}}
)
print(result["messages"][-1].content)

There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [ ]:
from langchain_core.tools import tool


@tool
def check_haiku_lines(text: str):
    """
    检查给定的俳句文本是否正好有3行。
    如果正确则返回None，否则返回错误消息。
    """
    #首先构建循环,首先使用strip()函数将text的首尾空白字符去除
    #使用splitlines()对清理后的text按行做分割
    #经过处理后的text检查每一行,如果不为空则清理空白字符串后放入列表中
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [ ]:
agent = create_agent(
    model="deepseek-chat",
    tools=[check_haiku_lines],
    system_prompt="你是一位只写俳句的体育诗人。你总会检查自己的作品。",
)

In [14]:
result = agent.invoke({"messages": "请给我写一首诗"})

checking haiku, it has 3 lines:
 篮球场上飞
汗水滴落如雨下
胜利的欢呼


In [15]:
result["messages"][-1].content

'**篮球场上飞**\n**汗水滴落如雨下**\n**胜利的欢呼**\n\n这首俳句描绘了篮球比赛的场景：运动员在场上飞奔，汗水如雨般洒落，最终迎来胜利的欢呼。'

In [16]:
print(len(result["messages"]))

4


In [17]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

请给我写一首诗
================================== Ai Message ==================================

作为一名只写俳句的体育诗人，我将为您创作一首体育主题的俳句。

让我先检查一下我的作品是否符合俳句的格式要求：
Tool Calls:
  check_haiku_lines (call_00_AftD5Tf33Hod5P7kvyX5f4FA)
 Call ID: call_00_AftD5Tf33Hod5P7kvyX5f4FA
  Args:
    text: 篮球场上飞
汗水滴落如雨下
胜利的欢呼
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

**篮球场上飞**
**汗水滴落如雨下**
**胜利的欢呼**

这首俳句描绘了篮球比赛的场景：运动员在场上飞奔，汗水如雨般洒落，最终迎来胜利的欢呼。


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [18]:
result

{'messages': [HumanMessage(content='请给我写一首诗', additional_kwargs={}, response_metadata={}, id='a17fc458-f01b-4976-a4bd-bd4ec833a320'),
  AIMessage(content='作为一名只写俳句的体育诗人，我将为您创作一首体育主题的俳句。\n\n让我先检查一下我的作品是否符合俳句的格式要求：', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 341, 'total_tokens': 434, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 341}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '60ec98ff-03e4-4c84-9b82-de81002cb1de', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--ad8bdd7f-8fd3-4344-becd-91ae4d613708-0', tool_calls=[{'name': 'check_haiku_lines', 'args': {'text': '篮球场上飞\n汗水滴落如雨下\n胜利的欢呼'}, 'id': 'call_00_AftD5Tf33Hod5P7kvyX5f4FA', 'type': 'tool_call'}], usage_metadata={'input_tokens': 341, 'output_tokens': 93,

You can select just the last message, and you can see where the final message is coming from.

In [19]:
result["messages"][-1]

AIMessage(content='**篮球场上飞**\n**汗水滴落如雨下**\n**胜利的欢呼**\n\n这首俳句描绘了篮球比赛的场景：运动员在场上飞奔，汗水如雨般洒落，最终迎来胜利的欢呼。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 460, 'total_tokens': 506, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 76}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '5629cc6b-add2-45d5-9bcb-68f3a76e072f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--82f65eba-a54b-45a8-9c5c-4fb352dad44c-0', usage_metadata={'input_tokens': 460, 'output_tokens': 46, 'total_tokens': 506, 'input_token_details': {'cache_read': 384}, 'output_token_details': {}})

In [20]:
result["messages"][-1].usage_metadata

{'input_tokens': 460,
 'output_tokens': 46,
 'total_tokens': 506,
 'input_token_details': {'cache_read': 384},
 'output_token_details': {}}

In [21]:
result["messages"][-1].response_metadata

{'token_usage': {'completion_tokens': 46,
  'prompt_tokens': 460,
  'total_tokens': 506,
  'completion_tokens_details': None,
  'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
  'prompt_cache_hit_tokens': 384,
  'prompt_cache_miss_tokens': 76},
 'model_provider': 'deepseek',
 'model_name': 'deepseek-chat',
 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache',
 'id': '5629cc6b-add2-45d5-9bcb-68f3a76e072f',
 'finish_reason': 'stop',
 'logprobs': None}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [ ]:
agent = create_agent(
    model="openai:gpt-5",
    tools=[check_haiku_lines],
    system_prompt="Your SYSTEM prompt here",
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()